# Flow Matching Neural Processes — A Complete Tutorial

This notebook is a self-contained, in-depth tutorial on the **Flow Matching Neural Process (FlowNP)** model. It walks through every idea, equation, and implementation choice — from the foundational papers to the final line of code.

### Prerequisite papers

| Paper | Key contribution |
|-------|-----------------|
| *Neural Processes* (Garnelo et al., 2018) | Defines the NP family: learn to map context sets to predictive distributions. |
| *Flow Matching for Generative Modelling* (Lipman et al., 2023) | Simulation-free training of continuous normalising flows via velocity regression. |
| *Flow Matching Neural Processes* (NeurIPS 2025) | Replaces the Gaussian head of NPs with a conditional flow-matching generative model. |

### Outline

1. [Neural Processes: The Meta-Learning Framework](#1)
2. [Flow Matching: Simulation-Free Generative Modelling](#2)
3. [FlowNP = NP + Flow Matching](#3)
4. [Architecture Deep-Dive](#4)
   - 4.1 Fourier positional encoding
   - 4.2 Token construction & time conditioning
   - 4.3 Embedding MLP
   - 4.4 Transformer encoder
   - 4.5 Velocity prediction head
   - 4.6 Padding masks for variable-length data
5. [Training: The CFM-MSE Objective](#5)
6. [Inference: Stochastic Euler Sampling](#6)
7. [Implementation Walk-Through (Code)](#7)
8. [Original vs. Clean Implementation — Key Differences](#8)
9. [Computational Cost & Performance Analysis](#9)
10. [Hands-On: Building and Running the Model](#10)

In [1]:
import sys, os
os.chdir(os.path.dirname(os.path.abspath("flownp_tutorial.ipynb"))
         if os.path.exists("flownp_tutorial.ipynb")
         else "/home/srahimis/SPL-Baseline-Neurips2026/flownp_spl")

import math
import numpy as np
import torch
import torch.nn as nn
import yaml
from torch.utils.data import DataLoader

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from data import SpatialDataset, spatial_collate
from model import FlowNP, fourier_encode, _build_mlp

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cpu")
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")

PyTorch 2.10.0 | Device: cpu


<a id="1"></a>
## 1. Neural Processes: The Meta-Learning Framework

### 1.1 Problem setup

Consider a set of **context** observations $\mathcal{C} = \{(x_c^{(i)}, y_c^{(i)})\}_{i=1}^{N_c}$ drawn from some unknown function $f$. Given new **target** locations $\{x_t^{(j)}\}_{j=1}^{N_t}$, our goal is to predict the corresponding values $\{y_t^{(j)}\}$ — including **uncertainty** over what those values might be.

This is the *few-shot function prediction* problem. Classical approaches include Gaussian Processes (GPs), but they scale as $\mathcal{O}(N^3)$ and require choosing a kernel. Neural Processes learn this mapping from data.

### 1.2 The Neural Process idea

A Neural Process defines a conditional distribution:

$$p_\theta(y_t \mid x_t, \mathcal{C})$$

It uses a neural network to:
1. **Encode** the context set into a representation.
2. **Decode** predictions at target locations conditioned on that representation.

The key insight is that NPs are **trained on many tasks** (each task = one context/target split), so they learn a prior over functions from data itself.

### 1.3 NP variants and their limitations

| Model | Decoder | Limitation |
|-------|---------|------------|
| **CNP** (Conditional NP) | Deterministic mean + variance | Cannot capture multi-modal distributions |
| **NP** (Latent NP) | Gaussian latent $z$ + decoder | Underfitting due to amortisation gap |
| **ANP** (Attentive NP) | Cross-attention + latent | Better, but still Gaussian output |
| **TNP** (Transformer NP) | Full Transformer | Powerful encoder, but Gaussian head limits expressivity |

**All of the above use a Gaussian output head** — they predict $\mu, \sigma$ at each target. This means they cannot represent multi-modal, skewed, or heavy-tailed predictive distributions.

FlowNP replaces this Gaussian head with a **flow-matching generative model**, enabling it to represent *any* conditional distribution.

In [2]:
# Visual: The NP meta-learning setup
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

for ax_idx, (n_ctx, title) in enumerate([(5, "Few context points"),
                                          (15, "More context"),
                                          (30, "Dense context")]):
    ax = axes[ax_idx]
    x_all = np.linspace(-3, 3, 200)
    y_true = np.sin(x_all) + 0.3 * np.cos(3 * x_all)

    rng = np.random.RandomState(42)
    idx = rng.choice(len(x_all), n_ctx, replace=False)
    xc, yc = x_all[idx], y_true[idx] + rng.randn(n_ctx) * 0.1

    ax.plot(x_all, y_true, "k-", alpha=0.3, label="True function")
    ax.scatter(xc, yc, c="#2563eb", s=40, zorder=5, label="Context")
    ax.fill_between(x_all, y_true - 0.5, y_true + 0.5,
                    color="#93c5fd", alpha=0.3, label="Uncertainty")
    ax.set_title(title, fontsize=11)
    ax.set_xlim(-3, 3)
    ax.set_ylim(-2.5, 2.5)
    if ax_idx == 0:
        ax.legend(fontsize=8, loc="upper left")


fig.suptitle("Neural Process Task: Predict function values at target locations given context",
             fontsize=12, y=1.02)
fig.tight_layout()
plt.savefig("np_concept.png", dpi=120, bbox_inches="tight")
plt.show()

/var/folders/3b/7ghp70vx3n74fxrjlqhnz64c0000gn/T/ipykernel_53750/2643327195.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<a id="2"></a>
## 2. Flow Matching: Simulation-Free Generative Modelling

### 2.1 Continuous Normalising Flows (CNFs)

A CNF defines a diffeomorphism between a simple base distribution $p_0$ (e.g. Gaussian) and the data distribution $p_1$ through an ODE:

$$\frac{dx_t}{dt} = v_\theta(x_t, t), \qquad t \in [0, 1]$$

Integrating from $t=0$ to $t=1$ transforms a noise sample $x_0 \sim p_0$ into a data sample $x_1 \sim p_1$.

**Problem**: Training CNFs traditionally requires solving the ODE (or its adjoint) during training, which is expensive and numerically fragile.

### 2.2 The Flow Matching objective

Lipman et al. (2023) showed that you can train the velocity field $v_\theta$ **without ever solving the ODE** during training. The key insight is to define a simple **probability path** from noise to data:

$$p_t(x) = [(1-t) \cdot p_0 + t \cdot p_1](x)$$

The simplest choice is the **linear interpolation path** (also called the optimal transport path):

$$x_t = t \cdot x_1 + (1 - t) \cdot x_0, \qquad x_0 \sim \mathcal{N}(0, I), \quad x_1 \sim p_{\text{data}}$$

The velocity that generates this path is simply:

$$u_t(x_t \mid x_0, x_1) = x_1 - x_0$$

This is a **constant** velocity — it doesn't depend on $t$!

### 2.3 The CFM loss

The **Conditional Flow Matching (CFM)** loss trains $v_\theta$ to match this target velocity:

$$\mathcal{L}_{\text{CFM}} = \mathbb{E}_{t \sim \mathcal{U}(0,1),\; x_0 \sim p_0,\; x_1 \sim p_1}\left[\left\| v_\theta(x_t, t) - (x_1 - x_0) \right\|^2\right]$$

This is a **simple MSE regression** — no ODE solving, no divergence estimation, no min-max optimisation.

### 2.4 Why this works

Lipman et al. prove that minimising the CFM loss is equivalent (in expectation) to minimising the Flow Matching loss that directly regresses against the marginal velocity field. The conditional formulation is tractable because we know both endpoints $x_0, x_1$ during training.

In [4]:
# Visual: The linear interpolation path from noise to data
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
times = [0.0, 0.25, 0.5, 0.75, 1.0]

rng = np.random.RandomState(7)
x0_samples = rng.randn(300, 2) * 0.8
theta = rng.uniform(0, 2 * np.pi, 300)
r = 2.0 + rng.randn(300) * 0.15
x1_samples = np.stack([r * np.cos(theta), r * np.sin(theta)], axis=1)

for ax, t_val in zip(axes, times):
    xt = t_val * x1_samples + (1 - t_val) * x0_samples
    ax.scatter(xt[:, 0], xt[:, 1], s=3, alpha=0.5, c="#2563eb")
    ax.set_title(f"t = {t_val:.2f}", fontsize=11)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.2)

fig.suptitle("Linear interpolation path: Gaussian noise (t=0) → data ring (t=1)",
             fontsize=12, y=1.02)
fig.tight_layout()
plt.savefig("flow_path.png", dpi=120, bbox_inches="tight")
plt.show()
print("The velocity at each point is simply v = x₁ − x₀ (constant in t).")

The velocity at each point is simply v = x₁ − x₀ (constant in t).


<a id="3"></a>
## 3. FlowNP = Neural Process + Flow Matching

### 3.1 The core idea

FlowNP replaces the Gaussian output head of a Neural Process with a **conditional flow-matching** generative model. Instead of predicting $\mu, \sigma$ directly, it learns a **conditional velocity field**:

$$v_\theta(y_t, t \mid \mathcal{C}, x_t)$$

where:
- $y_t$ is the noisy target value at flow-time $t$
- $\mathcal{C} = \{(x_c^{(i)}, y_c^{(i)})\}$ is the context set
- $x_t$ are the target spatial locations

### 3.2 Training objective

The FlowNP training loss is the **conditional CFM-MSE**, conditioned on the context:

$$\mathcal{L} = \mathbb{E}_{\mathcal{C}, x_t}\left[\mathbb{E}_{t, y_0, y_1}\left[\left\| v_\theta(y_t, t \mid \mathcal{C}, x_t) - (y_1 - y_0) \right\|^2\right]\right]$$

where $y_1$ is the true target value, $y_0 \sim \mathcal{N}(0, I)$, $y_t = t \cdot y_1 + (1-t) \cdot y_0$, and the outer expectation is over tasks (context/target splits from the dataset).

### 3.3 Inference

At test time, given a new context set and target locations, FlowNP generates predictions by integrating the learned velocity from noise:

$$y_0 \sim \mathcal{N}(0, I) \qquad \longrightarrow \qquad y_1 = y_0 + \int_0^1 v_\theta(y_t, t \mid \mathcal{C}, x_t)\, dt$$

This integral is approximated with an Euler scheme (discussed in Section 6).

### 3.4 Advantages over Gaussian-head NPs

| Property | Gaussian NPs | FlowNP |
|----------|:------------:|:------:|
| Multi-modal predictions | No | **Yes** |
| Skewed / heavy-tailed distributions | No | **Yes** |
| Correlated target predictions | Only via latent $z$ | **Naturally** (joint generation) |
| Training objective | NLL or ELBO | **Simple MSE** (no KL, no ELBO) |
| Sampling | Single forward pass | Iterative (T steps) |

<a id="4"></a>
## 4. Architecture Deep-Dive

The FlowNP architecture processes context and target information through a pipeline of five stages. Below is the complete data-flow diagram:

```
 Context (xc, yc)                    Target (xt, yt_noisy, t)
 ─────────────────                   ────────────────────────
        │                                     │
  Append time=1                         Append time=t
  xc' = [xc ‖ 1]                       xt' = [xt ‖ t]
        │                                     │
  ┌─────▼──────┐                        ┌─────▼──────┐
  │  Fourier   │                        │  Fourier   │
  │  Encoding  │                        │  Encoding  │
  └─────┬──────┘                        └─────┬──────┘
        │                                     │
  Concat features                       Concat features
  [enc(xc') ‖ yc]                       [enc(xt') ‖ yt]
        │                                     │
        └──────────────┬──────────────────────┘
                       │
                 Concatenate along
                 sequence dimension
                       │
               ┌───────▼────────┐
               │  Embedding MLP │   (token_dim → d_model)
               │   (4 layers)   │
               └───────┬────────┘
                       │
               ┌───────▼────────┐
               │   Transformer  │   Full bidirectional attention
               │    Encoder     │   with padding mask
               │  (6 layers,    │
               │   4 heads)     │
               └───────┬────────┘
                       │
                 Extract target
                 positions only
                       │
               ┌───────▼────────┐
               │  Velocity Head │   (d_model → dim_feedforward → dim_y)
               │ (Linear→ReLU   │
               │  →Linear)      │
               └───────┬────────┘
                       │
                       ▼
              v_θ(yt, t | C, xt)
              shape: (B, Nt, Dy)
```

Let's examine each stage in detail.

### 4.1 Fourier Positional Encoding

Raw spatial coordinates (e.g. `[0.5, 0.3]`) are low-dimensional and smooth. Neural networks struggle to learn high-frequency patterns from such inputs — this is the **spectral bias** problem (Rahaman et al., 2019).

Fourier positional encoding lifts each coordinate into a high-dimensional space of sinusoidal features, enabling the network to distinguish nearby points and learn fine-grained spatial patterns.

**Definition.** For a scalar input $x$ and frequency index $k = 0, 1, \ldots, \frac{n_\text{freq}}{2} - 1$:

$$\text{FourierEncode}(x) = \bigl[\sin(\omega_0 x),\; \cos(\omega_0 x),\; \sin(\omega_1 x),\; \cos(\omega_1 x),\; \ldots\bigr]$$

where $\omega_k = \pi \cdot 2^{k-2}$.

For a vector $x \in \mathbb{R}^D$, the encoding is applied independently to each dimension and concatenated, giving output dimension $D \times n_\text{freq}$.

**Frequency spectrum.** With the default `dim_posenc=80`, we use 40 frequency pairs per dimension. The frequencies span:

$$\omega_k = \pi \cdot 2^{k-2}, \quad k = 0, \ldots, 39$$

ranging from $\omega_0 = \pi/4 \approx 0.79$ to $\omega_{39} = \pi \cdot 2^{37} \approx 4.3 \times 10^{11}$.

In [5]:
# Implementation of fourier_encode and demonstration
# ──────────────────────────────────────────────────

# Here is the exact implementation from model.py:
def fourier_encode_explained(x, n_freq):
    """
    x:      (..., D)  input coordinates
    n_freq: int       total features per dimension (must be even)
    
    Returns: (..., D * n_freq)
    """
    half = n_freq // 2                          # number of sin/cos pairs
    # Frequencies: ω_k = π · 2^(k-2)  for k = 0, 1, ..., half-1
    omega = math.pi * (2.0 ** torch.arange(half, dtype=x.dtype, device=x.device).sub(2))
    
    phase = x.unsqueeze(-1) * omega             # (..., D, half) — outer product
    encoded = torch.cat([phase.sin(),           # (..., D, half)
                         phase.cos()], dim=-1)  # (..., D, 2*half) = (..., D, n_freq)
    return encoded.flatten(-2)                  # (..., D * n_freq)

# Demonstrate on a 2D coordinate
x_demo = torch.tensor([[[0.5, 0.3]]])  # (1, 1, 2)
enc = fourier_encode_explained(x_demo, n_freq=80)
print(f"Input:   shape {x_demo.shape}  →  {x_demo.squeeze().tolist()}")
print(f"Output:  shape {enc.shape}     →  2 dims × 80 features = 160 total")
print(f"Range:   [{enc.min():.4f}, {enc.max():.4f}]")

# Show the frequency spectrum
half = 40
freqs = math.pi * (2.0 ** (torch.arange(half, dtype=torch.float32) - 2))
fig, ax = plt.subplots(figsize=(10, 3))
ax.semilogy(range(half), freqs.numpy(), "o-", markersize=4, color="#2563eb")
ax.set_xlabel("Frequency index k")
ax.set_ylabel("ω_k")
ax.set_title("Fourier encoding frequency spectrum: ω_k = π · 2^(k−2)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig("freq_spectrum.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Lowest freq:  ω_0 = {freqs[0]:.4f}")
print(f"Highest freq: ω_39 = {freqs[-1]:.4e}")

Input:   shape torch.Size([1, 1, 2])  →  [0.5, 0.30000001192092896]
Output:  shape torch.Size([1, 1, 160])     →  2 dims × 80 features = 160 total
Range:   [-1.0000, 1.0000]
Lowest freq:  ω_0 = 0.7854
Highest freq: ω_39 = 4.3178e+11


### 4.2 Token construction and time conditioning

FlowNP builds a single token sequence from both context and target points. The critical design choice is **how time enters the representation**.

**Context tokens** are at flow-time $t = 1$ because they are fully observed (they have "already arrived" at the data distribution). A constant `1` is appended to their spatial coordinates before encoding:

$$\text{ctx\_token}_i = \bigl[\text{FourierEncode}([x_c^{(i)}, 1]) \;\big\|\; y_c^{(i)}\bigr]$$

**Target tokens** carry the current flow-time $t \in [0, 1]$ as their time coordinate:

$$\text{tar\_token}_j = \bigl[\text{FourierEncode}([x_t^{(j)}, t_j]) \;\big\|\; y_t^{(j)}\bigr]$$

where $y_t^{(j)}$ is the **noisy** target value at flow-time $t_j$.

This means **both spatial position and time are jointly Fourier-encoded**, allowing the model to learn space-time interactions through the non-linear MLP and Transformer.

**Token dimension:**

$$d_{\text{token}} = (\text{dim\_x} + 1) \times \text{dim\_posenc} + \text{dim\_y}$$

For PeakWeather: $(2 + 1) \times 80 + 4 = 244$. For GDP: $(3 + 1) \times 80 + 3 = 323$.

In [6]:
# Demonstrate token construction step by step
B, Nc, Nt = 1, 3, 2   # 1 batch, 3 context, 2 target
dim_x, dim_y, dim_posenc = 2, 4, 80

xc = torch.randn(B, Nc, dim_x)   # context positions
yc = torch.randn(B, Nc, dim_y)   # context values
xt = torch.randn(B, Nt, dim_x)   # target positions
yt = torch.randn(B, Nt, dim_y)   # noisy target values
t  = torch.rand(B, Nt, 1)        # flow time

# Step 1: Append time coordinate
xc_with_time = torch.cat([xc, torch.ones(B, Nc, 1)], dim=-1)   # t=1 for context
xt_with_time = torch.cat([xt, t], dim=-1)                        # t=t for targets

print("Step 1 — Append time coordinate:")
print(f"  xc: {list(xc.shape)} + [1]  →  xc_with_time: {list(xc_with_time.shape)}  (time=1)")
print(f"  xt: {list(xt.shape)} + [t]  →  xt_with_time: {list(xt_with_time.shape)}  (time=t)")

# Step 2: Fourier encode
xc_enc = fourier_encode(xc_with_time, dim_posenc)
xt_enc = fourier_encode(xt_with_time, dim_posenc)

print(f"\nStep 2 — Fourier encode:")
print(f"  xc_with_time: {list(xc_with_time.shape)}  →  xc_enc: {list(xc_enc.shape)}")
print(f"  xt_with_time: {list(xt_with_time.shape)}  →  xt_enc: {list(xt_enc.shape)}")
print(f"  Formula: ({dim_x}+1) × {dim_posenc} = {(dim_x+1)*dim_posenc}")

# Step 3: Concatenate with features
ctx_tokens = torch.cat([xc_enc, yc], dim=-1)
tar_tokens = torch.cat([xt_enc, yt], dim=-1)

print(f"\nStep 3 — Concatenate with features:")
print(f"  ctx_tokens: [xc_enc ‖ yc] = [{xc_enc.shape[-1]} + {dim_y}] = {list(ctx_tokens.shape)}")
print(f"  tar_tokens: [xt_enc ‖ yt] = [{xt_enc.shape[-1]} + {dim_y}] = {list(tar_tokens.shape)}")

# Step 4: Full token sequence
tokens = torch.cat([ctx_tokens, tar_tokens], dim=1)
print(f"\nStep 4 — Concatenate along sequence dimension:")
print(f"  tokens: [{Nc} ctx + {Nt} tar] = {list(tokens.shape)}")
print(f"  Token dimension: {tokens.shape[-1]}")

Step 1 — Append time coordinate:
  xc: [1, 3, 2] + [1]  →  xc_with_time: [1, 3, 3]  (time=1)
  xt: [1, 2, 2] + [t]  →  xt_with_time: [1, 2, 3]  (time=t)

Step 2 — Fourier encode:
  xc_with_time: [1, 3, 3]  →  xc_enc: [1, 3, 240]
  xt_with_time: [1, 2, 3]  →  xt_enc: [1, 2, 240]
  Formula: (2+1) × 80 = 240

Step 3 — Concatenate with features:
  ctx_tokens: [xc_enc ‖ yc] = [240 + 4] = [1, 3, 244]
  tar_tokens: [xt_enc ‖ yt] = [240 + 4] = [1, 2, 244]

Step 4 — Concatenate along sequence dimension:
  tokens: [3 ctx + 2 tar] = [1, 5, 244]
  Token dimension: 244


### 4.3 Embedding MLP

The token dimension ($d_\text{token}$) is typically much larger than the Transformer's hidden size ($d_\text{model}$). The **embedding MLP** projects tokens down to $d_\text{model}$ dimensions.

It is a standard **ReLU MLP** with `emb_depth` layers (default 4):

```
Linear(d_token, d_model) → ReLU → Linear(d_model, d_model) → ReLU → Linear(d_model, d_model)
                          ↑ depth=4 means 4 linear layers total ↑
```

This non-linear projection is important — a single linear layer would not be able to mix the Fourier features effectively. The MLP learns which frequency combinations are relevant for the task.

### 4.4 Transformer Encoder

The embedded tokens are processed by a standard **PyTorch `TransformerEncoder`** with:
- **Full (bidirectional) self-attention**: every token can attend to every other token. Context tokens attend to targets and vice versa. This is crucial — it allows the model to reason about the relationship between what it observes (context) and what it must predict (targets).
- **`num_layers=6`** stacked encoder layers, each containing multi-head self-attention + feed-forward + layer norm + residual connections.
- **`nhead=4`** attention heads, each of dimension `d_model / nhead = 16`.
- **No causal mask**: unlike language models, there is no autoregressive structure. All tokens see all others.

After encoding, only the **target-position outputs** (the last $N_t$ tokens) are extracted for velocity prediction. The context-position outputs are discarded — they served as conditioning.

### 4.5 Velocity Prediction Head

The velocity head is a small 2-layer MLP:

```
Linear(d_model, dim_feedforward) → ReLU → Linear(dim_feedforward, dim_y)
```

It maps the Transformer's output for each target token to the predicted velocity vector $v_\theta \in \mathbb{R}^{d_y}$.

### 4.6 Padding Masks for Variable-Length Data

Our datasets have **variable numbers of observation points** per sample (e.g. PeakWeather: ~128 points, GDP: 64–220 points). When batching, shorter samples are zero-padded. We must ensure that:

1. **The Transformer ignores padded tokens** in its attention computation.
2. **The loss function ignores padded target positions.**

For (1), we construct a `src_key_padding_mask` for the Transformer:

```python
pad_mask = ~cat([context_mask, target_mask], dim=1)   # True = padded → ignore
```

This is a `(B, Nc+Nt)` boolean tensor passed to `TransformerEncoder`. Padded tokens get `-inf` attention weights, so they neither attend nor are attended to.

For (2), the loss averages only over valid target points using the `target_mask`.

In [7]:
# Instantiate the full model and trace tensor shapes through each stage
model = FlowNP(dim_x=2, dim_y=4, dim_posenc=80, d_model=64, emb_depth=4,
               dim_feedforward=128, nhead=4, num_layers=6, timesteps=100)

print("FlowNP sub-modules and parameter counts:\n")
print(f"{'Module':<22s}  {'Type':<28s}  {'Parameters':>12s}")
print("─" * 66)
total = 0
for name, mod in model.named_children():
    n = sum(p.numel() for p in mod.parameters())
    total += n
    print(f"{name:<22s}  {type(mod).__name__:<28s}  {n:>12,}")
print("─" * 66)
print(f"{'TOTAL':<22s}  {'':28s}  {total:>12,}")

print(f"\n\nDetailed layer shapes for PeakWeather (dim_x=2, dim_y=4):")
token_dim = (2 + 1) * 80 + 4
print(f"  Token dimension:   ({2}+1) × {80} + {4} = {token_dim}")
print(f"  Embedder input:    {token_dim}")
print(f"  Embedder output:   {64}  (d_model)")
print(f"  Transformer:       {64} → 6 layers × (4-head attn + FFN(128)) → {64}")
print(f"  Velocity head:     {64} → 128 → {4}  (dim_y)")

FlowNP sub-modules and parameter counts:

Module                  Type                            Parameters
──────────────────────────────────────────────────────────────────
embedder                Sequential                          28,160
encoder                 TransformerEncoder                 200,832
velocity_head           Sequential                           8,836
──────────────────────────────────────────────────────────────────
TOTAL                                                      237,828


Detailed layer shapes for PeakWeather (dim_x=2, dim_y=4):
  Token dimension:   (2+1) × 80 + 4 = 244
  Embedder input:    244
  Embedder output:   64  (d_model)
  Transformer:       64 → 6 layers × (4-head attn + FFN(128)) → 64
  Velocity head:     64 → 128 → 4  (dim_y)


In [8]:
# Trace a forward pass with shape annotations at every stage
data = np.load("/home/zross/proj/spl/datasets/peakweather_irregular.npz", allow_pickle=True)
ds = SpatialDataset(data["x_train"][:20], data["pos_train"][:20])
loader = DataLoader(ds, batch_size=4, collate_fn=spatial_collate, drop_last=True)
batch = next(iter(loader))

xc, yc = batch["x_context"], batch["y_context"]
xt, yt = batch["x_target"],  batch["y_target"]
ctx_mask = batch["context_mask"]
tar_mask = batch["target_mask"]
B, Nc = xc.shape[:2]
Nt = xt.shape[1]

print(f"Input shapes (B={B}, Nc={Nc}, Nt={Nt}, Dx=2, Dy=4):")
print(f"  xc:       {list(xc.shape)}")
print(f"  yc:       {list(yc.shape)}")
print(f"  xt:       {list(xt.shape)}")
print(f"  yt:       {list(yt.shape)}")
print(f"  ctx_mask: {list(ctx_mask.shape)}  (True counts: {ctx_mask.sum(1).tolist()})")
print(f"  tar_mask: {list(tar_mask.shape)}  (True counts: {tar_mask.sum(1).tolist()})")

# Manually trace _velocity
t = torch.rand(B, Nt, 1)
xc_t = torch.cat([xc, xc.new_ones(B, Nc, 1)], dim=-1)
xt_t = torch.cat([xt, t], dim=-1)
print(f"\nAfter time append:")
print(f"  xc_t: {list(xc_t.shape)}  (last dim = {xc_t.shape[-1]})")
print(f"  xt_t: {list(xt_t.shape)}  (last dim = {xt_t.shape[-1]})")

xc_enc = fourier_encode(xc_t, 80)
xt_enc = fourier_encode(xt_t, 80)
print(f"\nAfter Fourier encoding (n_freq=80):")
print(f"  xc_enc: {list(xc_enc.shape)}  (3 × 80 = 240)")
print(f"  xt_enc: {list(xt_enc.shape)}")

ctx_tok = torch.cat([xc_enc, yc], dim=-1)
tar_tok = torch.cat([xt_enc, yt], dim=-1)
tokens = torch.cat([ctx_tok, tar_tok], dim=1)
print(f"\nToken assembly:")
print(f"  ctx_tokens: {list(ctx_tok.shape)}  (240 + 4 = 244)")
print(f"  tar_tokens: {list(tar_tok.shape)}")
print(f"  all_tokens: {list(tokens.shape)}  (Nc+Nt = {Nc+Nt})")

embedded = model.embedder(tokens)
print(f"\nAfter embedding MLP:")
print(f"  embedded: {list(embedded.shape)}  (244 → 64)")

pad_mask = ~torch.cat([ctx_mask, tar_mask], dim=1)
hidden = model.encoder(embedded, src_key_padding_mask=pad_mask)
print(f"\nAfter Transformer encoder:")
print(f"  hidden: {list(hidden.shape)}  (6 layers of self-attention)")

target_hidden = hidden[:, Nc:]
velocity = model.velocity_head(target_hidden)
print(f"\nExtract target outputs & velocity head:")
print(f"  target_hidden: {list(target_hidden.shape)}  (last {Nt} tokens)")
print(f"  velocity:      {list(velocity.shape)}  (64 → 128 → 4)")

Input shapes (B=4, Nc=60, Nt=97, Dx=2, Dy=4):
  xc:       [4, 60, 2]
  yc:       [4, 60, 4]
  xt:       [4, 97, 2]
  yt:       [4, 97, 4]
  ctx_mask: [4, 60]  (True counts: [31, 47, 60, 41])
  tar_mask: [4, 97]  (True counts: [97, 81, 68, 87])

After time append:
  xc_t: [4, 60, 3]  (last dim = 3)
  xt_t: [4, 97, 3]  (last dim = 3)

After Fourier encoding (n_freq=80):
  xc_enc: [4, 60, 240]  (3 × 80 = 240)
  xt_enc: [4, 97, 240]

Token assembly:
  ctx_tokens: [4, 60, 244]  (240 + 4 = 244)
  tar_tokens: [4, 97, 244]
  all_tokens: [4, 157, 244]  (Nc+Nt = 157)

After embedding MLP:
  embedded: [4, 157, 64]  (244 → 64)

After Transformer encoder:
  hidden: [4, 157, 64]  (6 layers of self-attention)

Extract target outputs & velocity head:
  target_hidden: [4, 97, 64]  (last 97 tokens)
  velocity:      [4, 97, 4]  (64 → 128 → 4)


<a id="5"></a>
## 5. Training: The CFM-MSE Objective

### 5.1 The complete training algorithm

For each mini-batch of context/target pairs:

**Algorithm: FlowNP Training Step**

> **Input:** Context $(x_c, y_c)$, target $(x_t, y_t)$, masks
>
> 1. Sample base noise: $y_0 \sim \mathcal{N}(0, I)$ with shape $(B, N_t, D_y)$
> 2. Sample flow-time: $t \sim \mathcal{U}(0, 1)$ with shape $(B, N_t, 1)$ — **independently per target point**
> 3. Interpolate: $y_t = t \cdot y_1 + (1 - t) \cdot y_0$
> 4. Predict velocity: $v_\theta = \text{FlowNP}(x_c, y_c, x_t, y_t, t)$
> 5. Target velocity: $v^* = y_1 - y_0$
> 6. Loss: $\mathcal{L} = \frac{1}{|V|} \sum_{j \in V} \|v_\theta^{(j)} - v^{*(j)}\|^2$ where $V$ = valid target indices
> 7. Backpropagate and update parameters

### 5.2 Key design details

**Independent time per target point** (step 2): Each target point gets its own random $t$, not a shared one for the whole batch. This provides richer training signal — at each step, the model sees a mixture of easy (high-$t$, close to data) and hard (low-$t$, close to noise) targets.

**The target velocity is independent of $t$** (step 5): $v^* = y_1 - y_0$ is the same regardless of what $t$ we sampled. This is a property of the linear interpolation path.

**Masked loss** (step 6): Because of variable-length padding, we must only compute the loss over valid (non-padded) target points. The implementation normalises by the number of valid points per sample, then averages across the batch:

```python
sq_err = (v_pred - v_true).pow(2).mean(dim=-1)          # (B, Nt)
n_valid = tar_mask.float().sum(dim=1).clamp(min=1)       # (B,)
loss = ((sq_err * tar_mask.float()).sum(1) / n_valid).mean()  # scalar
```

In [9]:
# Step-by-step training forward pass with visualisation
model.train()

B_demo, Nt_demo, Dy_demo = 4, xt.shape[1], 4

# Step 1: Sample noise
y0 = torch.randn_like(yt)
print(f"Step 1 — y₀ ~ N(0,I): shape {list(y0.shape)}")

# Step 2: Sample time (independent per target)
t_train = torch.rand(B_demo, Nt_demo, 1)
print(f"Step 2 — t ~ U(0,1):  shape {list(t_train.shape)}")
print(f"         Example t values for sample 0: {t_train[0, :5, 0].tolist()[:5]}")

# Step 3: Interpolate
yt_noisy = t_train * yt + (1.0 - t_train) * y0
print(f"Step 3 — yₜ = t·y₁ + (1-t)·y₀: shape {list(yt_noisy.shape)}")

# Step 4: Predict velocity
v_pred = model._velocity(xc, yc, xt, yt_noisy, t_train,
                         ctx_mask=ctx_mask, tar_mask=tar_mask)
print(f"Step 4 — v_θ predicted: shape {list(v_pred.shape)}")

# Step 5: Target velocity
v_true = yt - y0
print(f"Step 5 — v* = y₁ - y₀: shape {list(v_true.shape)}")

# Step 6: Masked loss
sq_err = (v_pred - v_true).pow(2).mean(dim=-1)
n_valid = tar_mask.float().sum(dim=1).clamp(min=1)
loss = ((sq_err * tar_mask.float()).sum(dim=1) / n_valid).mean()
print(f"Step 6 — Masked MSE loss: {loss.item():.6f}")

# Visualise: how yt changes with t for one target point
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
y1_single = yt[0, 0].detach().numpy()  # one target, 4 features
y0_single = y0[0, 0].detach().numpy()
t_vis = np.linspace(0, 1, 50)

for ch, (ax, name) in enumerate(zip(axes, ["Temp", "Pressure", "Humidity", "Wind"])):
    yt_ch = [tv * y1_single[ch] + (1 - tv) * y0_single[ch] for tv in t_vis]
    ax.plot(t_vis, yt_ch, color="#2563eb", linewidth=2)
    ax.axhline(y1_single[ch], color="#16a34a", linestyle="--", alpha=0.7, label=f"y₁={y1_single[ch]:.2f}")
    ax.axhline(y0_single[ch], color="#dc2626", linestyle="--", alpha=0.7, label=f"y₀={y0_single[ch]:.2f}")
    ax.set_xlabel("t")
    ax.set_title(name, fontsize=10)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.2)

fig.suptitle("Linear interpolation path yₜ = t·y₁ + (1−t)·y₀ for one target point", fontsize=11)
fig.tight_layout()
plt.savefig("interpolation_path.png", dpi=120, bbox_inches="tight")
plt.show()

Step 1 — y₀ ~ N(0,I): shape [4, 97, 4]
Step 2 — t ~ U(0,1):  shape [4, 97, 1]
         Example t values for sample 0: [0.992743730545044, 0.6075905561447144, 0.39700233936309814, 0.9188354015350342, 0.4221540689468384]
Step 3 — yₜ = t·y₁ + (1-t)·y₀: shape [4, 97, 4]
Step 4 — v_θ predicted: shape [4, 97, 4]
Step 5 — v* = y₁ - y₀: shape [4, 97, 4]
Step 6 — Masked MSE loss: 2.446615


<a id="6"></a>
## 6. Inference: Stochastic Euler Sampling

### 6.1 The sampling algorithm

At inference, we generate predictions by integrating the learned velocity field from Gaussian noise. The FlowNP paper uses a **stochastic Euler scheme** with time-dependent noise injection:

**Algorithm: FlowNP Prediction**

> **Input:** Context $(x_c, y_c)$, target locations $x_t$, number of samples $S$, timesteps $T$
>
> 1. For $s = 1, \ldots, S$ (independent samples):
>    - Initialise $y_0^{(s)} \sim \mathcal{N}(0, I)$
>    - For $k = 0, 1, \ldots, T-1$:
>      - $t = k / T$
>      - $v = v_\theta(y_t^{(s)}, t \mid \mathcal{C}, x_t)$ — velocity prediction
>      - $\alpha(t) = 1 + t(1-t)$ — velocity scaling
>      - $\sigma(t) = 0.2\sqrt{t(1-t)}$ — noise level
>      - $y_{t+\Delta t}^{(s)} = y_t^{(s)} + \bigl(\alpha(t) \cdot v + \sigma(t) \cdot \epsilon\bigr) \cdot \Delta t$, where $\epsilon \sim \mathcal{N}(0, I)$
> 2. Return mean and std across the $S$ samples

### 6.2 Why stochastic Euler?

The **deterministic Euler** scheme ($\sigma = 0$) would be:

$$y_{t+\Delta t} = y_t + v_\theta(y_t, t) \cdot \Delta t$$

The stochastic version adds controlled noise, which serves two purposes:

1. **Sample diversity**: Without noise, all $S$ trajectories starting from different $y_0$ might collapse to similar predictions (especially with an over-trained model). Noise maintains diversity.

2. **Exploration**: Noise helps the integration explore the predictive distribution, giving better uncertainty estimates.

### 6.3 The time-dependent coefficients

The coefficients $\alpha(t)$ and $\sigma(t)$ are designed to:

- **$\alpha(t) = 1 + t(1-t)$**: Scales the velocity *above* 1.0 in the middle of the flow. This accelerates transport where the distribution is most uncertain. At the boundaries ($t=0, t=1$), $\alpha = 1$, recovering standard Euler.

- **$\sigma(t) = 0.2\sqrt{t(1-t)}$**: Noise is **zero at the boundaries** (no perturbation when starting from pure noise or arriving at data) and maximal at $t=0.5$ (peak uncertainty). The factor 0.2 keeps noise moderate.

### 6.4 Computational cost of sampling

Each sample requires $T$ forward passes through the model. With $S$ samples and batch size $B$, the total cost is:

$$\text{FLOPs}_\text{predict} = S \times T \times \text{FLOPs}_\text{forward}$$

For our defaults ($S=20, T=100$), this is **2000× the cost of a single training step**. This is why the sample-based evaluation is much slower than the CFM loss evaluation (which needs only 1 forward pass per batch).

In [14]:
# Visualise the sampling process step by step
model.eval()
model.timesteps = 20  # Reduced for visualisation speed

B_s, Nt_s = 1, xt.shape[1]
Dy_s = 4
S_vis = 1  # Single trajectory for visualisation

xc_s = xc[:1]
yc_s = yc[:1]
xt_s = xt[:1]
cm_s = ctx_mask[:1]
tm_s = tar_mask[:1]

# Track one target point's trajectory through the flow
trajectories = []
yt_s = torch.randn(1, Nt_s, Dy_s)
T_s = model.timesteps

for step in range(T_s):
    s_val = step / T_s
    trajectories.append(yt_s[0, 0].detach().numpy())  # Track point 0
    
    t_vec = yt_s.new_full((1, Nt_s, 1), s_val)
    v = model._velocity(xc_s, yc_s, xt_s, yt_s, t_vec, ctx_mask=cm_s, tar_mask=tm_s)
    
    alpha = 1.0 + s_val * (1.0 - s_val)
    sigma = 0.2 * math.sqrt(max(s_val * (1.0 - s_val), 0.0))
    yt_s = yt_s + (alpha * v + sigma * torch.randn_like(yt_s)) / T_s

trajectories.append(yt_s[0, 0].detach().numpy())
trajectories = np.array(trajectories)

# Plot: trajectory of one point through the flow
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
t_axis = np.linspace(0, 1, len(trajectories))
channel_names = ["Temperature", "Pressure", "Humidity", "Wind Speed"]
colors = ["#2563eb", "#dc2626", "#16a34a", "#f59e0b"]

for ch, (ax, name, color) in enumerate(zip(axes, channel_names, colors)):
    ax.plot(t_axis, trajectories[:, ch], color=color, linewidth=2)
    ax.axhline(yt[0, 0, ch].item(), color="k", linestyle="--", alpha=0.5,
               label=f"Ground truth: {yt[0, 0, ch].item():.2f}")
    ax.scatter([0], [trajectories[0, ch]], c=color, s=50, zorder=5, marker="o")
    ax.scatter([1], [trajectories[-1, ch]], c=color, s=50, zorder=5, marker="*")
    ax.set_xlabel("Flow time t")
    ax.set_ylabel(name)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.2)

fig.suptitle("Euler integration trajectory for one target point (noise → prediction)", fontsize=11)
fig.tight_layout()
plt.savefig("euler_trajectory.png", dpi=120, bbox_inches="tight")
plt.show()

# Also plot alpha(t) and sigma(t)
t_plot = np.linspace(0, 1, 200)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

ax1.plot(t_plot, 1 + t_plot * (1 - t_plot), color="#2563eb", linewidth=2)
ax1.set_title("α(t) = 1 + t(1−t)  [velocity scaling]")
ax1.set_xlabel("t"); ax1.set_ylabel("α(t)")
ax1.axhline(1.0, color="gray", linestyle=":", alpha=0.5)
ax1.annotate("peak at t=0.5:\nα=1.25", xy=(0.5, 1.25), fontsize=9,
             ha="center", va="bottom")
ax1.grid(True, alpha=0.2)

ax2.plot(t_plot, 0.2 * np.sqrt(np.maximum(t_plot * (1 - t_plot), 0)),
         color="#dc2626", linewidth=2)
ax2.set_title("σ(t) = 0.2·√(t(1−t))  [noise injection]")
ax2.set_xlabel("t"); ax2.set_ylabel("σ(t)")
ax2.annotate("peak at t=0.5:\nσ=0.1", xy=(0.5, 0.1), fontsize=9,
             ha="center", va="bottom")
ax2.grid(True, alpha=0.2)

fig.tight_layout()
plt.savefig("stochastic_euler_coeffs.png", dpi=120, bbox_inches="tight")
plt.show()

model.timesteps = 100  # Reset

<a id="7"></a>
## 7. Implementation Walk-Through (Code)

This section walks through the actual Python code in `model.py`, mapping each
function and method to the mathematical concepts from the previous sections.

### 7.1 `fourier_encode` (lines 25–38)

```python
def fourier_encode(x: torch.Tensor, n_freq: int) -> torch.Tensor:
    half = n_freq // 2
    omega = math.pi * (2.0 ** torch.arange(half, ...).sub(2))
    phase = x.unsqueeze(-1) * omega       # (..., D, half)
    return torch.cat([phase.sin(), phase.cos()], dim=-1).flatten(-2)
```

**Maps to**: Section 4.1.  The `sub(2)` makes the first frequency `π·2⁻² = π/4`
rather than `π·2⁰ = π`, giving a gentler start to the frequency ladder.

### 7.2 `_build_mlp` (lines 41–50)

```python
def _build_mlp(in_dim, hid_dim, out_dim, depth):
    layers = [nn.Linear(in_dim, hid_dim), nn.ReLU(inplace=True)]
    for _ in range(depth - 2):
        layers += [nn.Linear(hid_dim, hid_dim), nn.ReLU(inplace=True)]
    layers.append(nn.Linear(hid_dim, out_dim))
    return nn.Sequential(*layers)
```

Used for both the **embedder** (depth=4) and could be reused for other MLPs.
Note: no output activation — the embedder feeds into a Transformer (which has
its own layer norm), and the velocity head must output unconstrained velocities.

### 7.3 `FlowNP.__init__` (lines 72–109)

Key dimension calculation:

```python
token_dim = (dim_x + 1) * dim_posenc + dim_y
```

The `+1` is the flow-time dimension appended to spatial coords (Section 4.2).

Three learnable modules:
- `self.embedder`: MLP mapping `token_dim → d_model`
- `self.encoder`: `nn.TransformerEncoder` (standard PyTorch)
- `self.velocity_head`: 2-layer MLP mapping `d_model → dim_y`

### 7.4 `FlowNP._velocity` (lines 117–149)

This is the **core forward computation**. The implementation follows the
data-flow diagram from Section 4 exactly:

1. Append time: `xc_t = [xc ‖ 1]`, `xt_t = [xt ‖ t]`
2. Fourier encode: `xc_enc`, `xt_enc`
3. Build tokens: `[enc ‖ features]` for each
4. Concatenate: `[context_tokens, target_tokens]`
5. Build padding mask: `~cat([ctx_mask, tar_mask])`
6. Embed → Encode → Extract target outputs → Velocity head

### 7.5 `FlowNP.forward` (lines 153–187) — Training

```python
y0 = torch.randn_like(y_target)                    # Step 1
t = torch.rand(B, Nt, 1, device=y_target.device)   # Step 2
yt = t * y_target + (1.0 - t) * y0                  # Step 3
v_pred = self._velocity(...)                         # Step 4
v_true = y_target - y0                               # Step 5
# Masked MSE                                        # Step 6
```

Note: `torch.rand` gives $t \in [0, 1)$, so the model never trains at
exactly $t=1$. This is fine — the flow is defined on the open interval.

### 7.6 `FlowNP.predict` (lines 191–234) — Inference

The prediction method:
1. Replicates inputs `S` times along the batch dimension for parallel sampling.
2. Initialises `yt` from Gaussian noise.
3. Runs `T` Euler steps with the stochastic scheme from Section 6.
4. Reshapes `(S*B, Nt, Dy) → (S, B, Nt, Dy)` and computes mean/std.

The `@torch.no_grad()` decorator disables gradient tracking for efficiency.

<a id="8"></a>
## 8. Original vs. Clean Implementation — Key Differences

Our `model.py` is a clean re-implementation of the original `flowNP/models/fnp.py`
from the paper's code release.  While the **mathematical algorithm is identical**,
the code structure differs significantly.  This section documents every difference
and why it was made.

### 8.1 Structural changes

| Aspect | Original (`flowNP/models/fnp.py`) | Clean (`flownp_spl/model.py`) |
|--------|-----------------------------------|-------------------------------|
| **Velocity field** | Nested `Predict_Velocity` class defined *inside* `__init__`, capturing `encode` as a closure | Proper `_velocity` method on the `FlowNP` class |
| **Encode function** | Closure `encode` defined inside `__init__` and assigned to `self.encode` | Integrated directly into `_velocity` — no separate function needed |
| **Padding support** | None — assumes fixed-size sequences (GP, ERA5 grid) | Full `src_key_padding_mask` support for variable-length data |
| **Masked loss** | `nn.MSELoss()` on all points | Manual masked MSE that excludes padded positions |
| **ODE solver** | `flow_matching.ODESolver` wired in at `__init__` time | Not included (sampling uses Euler; ODE solver can be added separately) |
| **Type annotations** | None | Full type hints on all public methods |
| **Batch interface** | `AttrDict` with `.xc`, `.yc`, `.xt`, `.yt` attributes | Explicit tensor arguments — clearer API |
| **Output format** | `AttrDict(loss=..., tar_ll=...)` | `dict[str, Tensor]` with `"loss"` key |

### 8.2 Algorithm preservation

Despite the structural changes, the **core algorithm is preserved exactly**:

1. **Same positional encoding**: `fourier_encode` produces identical output to the
   original `comp_posenc` — same frequency formula `ω_k = π · 2^(k-2)`.

2. **Same token construction**: context gets `time=1`, targets get `time=t`,
   both are Fourier-encoded and concatenated with features.

3. **Same training loss**: CFM-MSE: `‖v_pred − (y₁ − y₀)‖²` with linear path.

4. **Same Euler sampling**: identical stochastic coefficients `α(t)=1+t(1-t)`,
   `σ(t)=0.2√(t(1-t))`.

### 8.3 What the original code does differently in `predict`

One notable difference: the original `predict` method flows at **all positions**
(context + target), not just target positions:

```python
# Original: include context positions in the flow
xt = torch.cat((xc, xt.repeat(...)), dim=1)
yt = torch.randn(S*B, Nc+Nt, Dy)  # noise for ALL positions
# ... Euler integration ...
return yt[:, Nc:]  # return only target portion
```

This means the original model also generates (and discards) values at context
locations during inference.  Our clean implementation flows only at target
locations, which is:
- More consistent with training (which only flows targets)
- More computationally efficient (processes `Nt` instead of `Nc + Nt` flow tokens)
- Simpler to understand

The original approach may provide a slight quality boost through implicit
self-consistency (the model "checks" that context locations flow to correct values),
but this benefit is not established in the paper.

In [15]:
# Verify that our Fourier encoding matches the original
# ─────────────────────────────────────────────────────

def comp_posenc_original(dim_posenc, pos):
    """Exact copy of the original flowNP/models/fnp.py comp_posenc."""
    shp = pos.shape
    omega = torch.arange(dim_posenc // 2, dtype=torch.float).to(pos.device)
    omega = torch.pi * 2 ** (omega - 2)
    out = pos[:, :, :, None] * omega[None, None, None, :]
    emb_sin = torch.sin(out)
    emb_cos = torch.cos(out)
    emb = torch.concatenate([emb_sin, emb_cos], axis=-1)
    return emb.reshape(shp[0], shp[1], shp[-1] * dim_posenc)

# Test with a sample input
test_input = torch.randn(2, 5, 3)  # (B=2, N=5, D=3)
n_freq = 80

original_out = comp_posenc_original(n_freq, test_input)
clean_out = fourier_encode(test_input, n_freq)

max_diff = (original_out - clean_out).abs().max().item()
print(f"Maximum absolute difference between original and clean encoding: {max_diff:.2e}")
print(f"Match: {'YES' if max_diff < 1e-5 else 'NO'}")
print(f"Output shapes — original: {list(original_out.shape)}, clean: {list(clean_out.shape)}")

Maximum absolute difference between original and clean encoding: 0.00e+00
Match: YES
Output shapes — original: [2, 5, 240], clean: [2, 5, 240]


<a id="9"></a>
## 9. Computational Cost & Performance Analysis

### 9.1 Model size

The FlowNP model is relatively **lightweight** compared to modern Transformers.
With the default config (`d_model=64`, `num_layers=6`, `nhead=4`), the total
parameter count is ~240K — small enough to train on a single GPU in minutes.

In [16]:
# Parameter breakdown
model_w = FlowNP(dim_x=2, dim_y=4, dim_posenc=80, d_model=64, emb_depth=4,
                 dim_feedforward=128, nhead=4, num_layers=6, timesteps=100)
model_g = FlowNP(dim_x=3, dim_y=3, dim_posenc=80, d_model=64, emb_depth=4,
                 dim_feedforward=128, nhead=4, num_layers=6, timesteps=100)

print("Parameter breakdown by component:\n")
print(f"{'Component':<22s}  {'PeakWeather':>12s}  {'GDP':>12s}  Notes")
print("─" * 75)

for name in ["embedder", "encoder", "velocity_head"]:
    nw = sum(p.numel() for p in getattr(model_w, name).parameters())
    ng = sum(p.numel() for p in getattr(model_g, name).parameters())
    notes = ""
    if name == "embedder":
        notes = f"  token_dim differs: {(2+1)*80+4} vs {(3+1)*80+3}"
    print(f"{name:<22s}  {nw:>12,}  {ng:>12,}{notes}")

print("─" * 75)
print(f"{'Total':<22s}  {model_w.num_params:>12,}  {model_g.num_params:>12,}")
print(f"\nThe embedder is larger for GDP because token_dim is bigger (323 vs 244).")
print(f"The Transformer encoder and velocity head are identical between datasets.")

Parameter breakdown by component:

Component                PeakWeather           GDP  Notes
───────────────────────────────────────────────────────────────────────────
embedder                      28,160        33,216  token_dim differs: 244 vs 323
encoder                      200,832       200,832
velocity_head                  8,836         8,707
───────────────────────────────────────────────────────────────────────────
Total                        237,828       242,755

The embedder is larger for GDP because token_dim is bigger (323 vs 244).
The Transformer encoder and velocity head are identical between datasets.


### 9.2 Computational complexity

**Training (per step):** The dominant cost is the Transformer encoder's self-attention, which is $\mathcal{O}((N_c + N_t)^2 \cdot d_\text{model})$. For our datasets:

| Dataset | Typical $N_c + N_t$ | Attention cost factor |
|---------|:---------------------:|:---------------------:|
| PeakWeather | ~128 | $128^2 = 16{,}384$ |
| GDP | ~64–220 | $64^2$ to $220^2$ |

With `d_model=64` and 6 layers, the per-step cost is modest on modern GPUs.

**Inference (per batch):** Each of the $T=100$ Euler steps requires one full forward pass. With $S=20$ samples, the total is $20 \times 100 = 2{,}000$ forward passes per batch. This makes inference approximately **2000× slower than a single training step**.

### 9.3 Training performance (observed)

From our test runs with `batch_size=8` on CPU (2 epochs, reduced settings):

| Metric | PeakWeather | GDP |
|--------|:-----------:|:---:|
| Epoch 1 train loss | 0.829 | 1.146 |
| Epoch 2 train loss | 0.457 | 1.055 |
| Epoch 2 test loss | 0.426 | 0.986 |
| Final prediction MSE | 0.167 | 0.653 |
| Time per epoch (CPU) | ~76s | ~128s |

Key observations:
- **Loss drops rapidly** in the first epoch — the model quickly learns the broad structure.
- **PeakWeather converges faster** than GDP — likely because the weather data is more regular (fixed ~128 stations) while GDP has highly variable point counts and 3D geometry.
- **No overfitting in 2 epochs** — test loss tracks train loss closely, indicating more epochs would help.

### 9.4 Memory considerations

Peak memory usage is dominated by the Transformer's attention matrices:

$$\text{Memory} \propto B \times H \times (N_c + N_t)^2$$

where $H$ = number of heads. During prediction, the effective batch size is $S \times B$ (because samples are parallelised), which can be memory-intensive. If running out of GPU memory during prediction, reduce `num_samples` or `batch_size`.

### 9.5 Scaling tips

| To achieve | Adjust |
|------------|--------|
| Faster training | Reduce `num_layers` (4 is often sufficient), or reduce `d_model` to 32 |
| Better accuracy | Increase `d_model` to 128, `num_layers` to 8, train for more epochs |
| Faster inference | Reduce `timesteps` (50 often works), reduce `eval_num_samples` |
| Lower memory | Reduce `batch_size`, or reduce `dim_posenc` to 40 |

In [17]:
# Profile forward pass timing for different sequence lengths
import time as _time

model_prof = FlowNP(dim_x=2, dim_y=4, dim_posenc=80, d_model=64, emb_depth=4,
                    dim_feedforward=128, nhead=4, num_layers=6, timesteps=100)
model_prof.eval()

print(f"{'Sequence length':>16s}  {'Forward (ms)':>12s}  {'Predict T=10 (ms)':>18s}")
print("─" * 52)

for seq_len in [32, 64, 128, 256]:
    nc = seq_len // 3
    nt = seq_len - nc
    xc_p = torch.randn(4, nc, 2)
    yc_p = torch.randn(4, nc, 4)
    xt_p = torch.randn(4, nt, 2)
    yt_p = torch.randn(4, nt, 4)
    
    # Warm up
    _ = model_prof(xc_p, yc_p, xt_p, yt_p)
    
    # Time forward
    t0 = _time.perf_counter()
    for _ in range(5):
        _ = model_prof(xc_p, yc_p, xt_p, yt_p)
    fwd_ms = ((_time.perf_counter() - t0) / 5) * 1000
    
    # Time predict (reduced timesteps)
    model_prof.timesteps = 10
    t0 = _time.perf_counter()
    with torch.no_grad():
        _ = model_prof.predict(xc_p, yc_p, xt_p, num_samples=3)
    pred_ms = (_time.perf_counter() - t0) * 1000
    model_prof.timesteps = 100
    
    print(f"{seq_len:>16d}  {fwd_ms:>12.1f}  {pred_ms:>18.1f}")

print("\nForward time scales quadratically with sequence length (self-attention).")
print("Predict time ≈ num_samples × timesteps × forward time.")

 Sequence length  Forward (ms)   Predict T=10 (ms)
────────────────────────────────────────────────────
              32           3.1                52.8
              64           4.6                94.5
             128          11.6               253.2
             256          27.5               647.0

Forward time scales quadratically with sequence length (self-attention).
Predict time ≈ num_samples × timesteps × forward time.


<a id="10"></a>
## 10. Hands-On: Building and Running the Model

This final section puts everything together with a mini training run
directly in the notebook, demonstrating the full pipeline end-to-end.

In [18]:
import time as _time
from train import train_epoch, evaluate_loss, evaluate_prediction, _to_device

# ── Configuration ────────────────────────────────────────────────
cfg = {
    "data_path": "/home/zross/proj/spl/datasets/peakweather_irregular.npz",
    "x_dim": 2,  "y_dim": 4,
    "dim_posenc": 80,  "d_model": 64,  "emb_depth": 4,
    "dim_feedforward": 128,  "nhead": 4,  "num_layers": 6,
    "timesteps": 5,  # reduced for demo speed
    "learning_rate": 5e-4,
    "context_frac_range": [0.1, 0.5],
    "eval_context_frac": 0.3,
}

# ── Data ─────────────────────────────────────────────────────────
data = np.load(cfg["data_path"], allow_pickle=True)
train_ds = SpatialDataset(data["x_train"], data["pos_train"],
                          context_frac_range=tuple(cfg["context_frac_range"]))
test_ds  = SpatialDataset(data["x_test"], data["pos_test"],
                          context_frac_range=(cfg["eval_context_frac"],
                                              cfg["eval_context_frac"]),
                          deterministic=True)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          num_workers=0, collate_fn=spatial_collate, drop_last=True)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False,
                          num_workers=0, collate_fn=spatial_collate)

# ── Model ────────────────────────────────────────────────────────
demo_model = FlowNP(
    dim_x=cfg["x_dim"], dim_y=cfg["y_dim"], dim_posenc=cfg["dim_posenc"],
    d_model=cfg["d_model"], emb_depth=cfg["emb_depth"],
    dim_feedforward=cfg["dim_feedforward"], nhead=cfg["nhead"],
    num_layers=cfg["num_layers"], timesteps=cfg["timesteps"],
)

# ── Optimiser ────────────────────────────────────────────────────
optimizer = torch.optim.Adam(demo_model.parameters(), lr=cfg["learning_rate"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=2)

print(f"Model:  {demo_model.num_params:,} parameters")
print(f"Data:   {len(train_ds):,} train / {len(test_ds):,} test samples")
print(f"Config: batch_size=8, timesteps={cfg['timesteps']}, lr={cfg['learning_rate']}")

# ── Train 2 epochs ───────────────────────────────────────────────
history = {"epoch": [], "train_loss": [], "test_loss": []}
print(f"\n{'='*60}")

for epoch in range(1, 3):
    t0 = _time.time()
    train_loss = train_epoch(demo_model, train_loader, optimizer, DEVICE, report_freq=500)
    scheduler.step()
    test_loss = evaluate_loss(demo_model, test_loader, DEVICE)
    elapsed = _time.time() - t0

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["test_loss"].append(test_loss)
    
    print(f"Epoch {epoch}/2  train={train_loss:.4f}  test={test_loss:.4f}  "
          f"lr={scheduler.get_last_lr()[0]:.2e}  time={elapsed:.0f}s")

print(f"{'='*60}")

Model:  237,828 parameters
Data:   10,000 train / 1,000 test samples
Config: batch_size=8, timesteps=5, lr=0.0005

  step     0  loss=2.114627


KeyboardInterrupt: 

In [ ]:
# Generate predictions and visualise
demo_model.eval()

test_batch = next(iter(test_loader))
test_batch = _to_device(test_batch, DEVICE)

mean, std = demo_model.predict(
    test_batch["x_context"], test_batch["y_context"],
    test_batch["x_target"], num_samples=5,
    ctx_mask=test_batch["context_mask"],
    tar_mask=test_batch["target_mask"],
)

# Pick one sample from the batch and plot predictions vs ground truth
idx = 0
mask_ctx = test_batch["context_mask"][idx].bool()
mask_tar = test_batch["target_mask"][idx].bool()

xc_vis = test_batch["x_context"][idx][mask_ctx].numpy()
yc_vis = test_batch["y_context"][idx][mask_ctx].numpy()
xt_vis = test_batch["x_target"][idx][mask_tar].numpy()
yt_vis = test_batch["y_target"][idx][mask_tar].numpy()
pred_vis = mean[idx][mask_tar].numpy()
std_vis  = std[idx][mask_tar].numpy()

channel_names = ["Temperature", "Pressure", "Humidity", "Wind Speed"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ch, (ax, name) in enumerate(zip(axes.flat, channel_names)):
    # Sort by first coordinate for cleaner plots
    order_c = np.argsort(xc_vis[:, 0])
    order_t = np.argsort(xt_vis[:, 0])
    
    ax.scatter(xc_vis[order_c, 0], yc_vis[order_c, ch], c="#2563eb", s=20,
               alpha=0.7, label="Context", zorder=3)
    ax.scatter(xt_vis[order_t, 0], yt_vis[order_t, ch], c="#16a34a", s=15,
               alpha=0.5, label="Target (true)", zorder=2)
    ax.scatter(xt_vis[order_t, 0], pred_vis[order_t, ch], c="#dc2626", s=15,
               alpha=0.5, label="Prediction", marker="x", zorder=2)
    ax.fill_between(xt_vis[order_t, 0],
                    pred_vis[order_t, ch] - 2 * std_vis[order_t, ch],
                    pred_vis[order_t, ch] + 2 * std_vis[order_t, ch],
                    color="#fca5a5", alpha=0.3, label="±2σ")
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Coordinate dim 0")
    if ch == 0:
        ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

fig.suptitle("FlowNP Predictions vs Ground Truth (PeakWeather, 2-epoch model)",
             fontsize=12, y=1.01)
fig.tight_layout()
plt.savefig("predictions_demo.png", dpi=120, bbox_inches="tight")
plt.show()

# Report MSE
mse_per_ch = ((pred_vis - yt_vis) ** 2).mean(axis=0)
print("Per-channel prediction MSE (after only 2 epochs):")
for name, mse_val in zip(channel_names, mse_per_ch):
    print(f"  {name:15s}: {mse_val:.4f}")

## Summary & Key Takeaways

### The Big Picture

FlowNP bridges two powerful ideas:

1. **Neural Processes** provide a meta-learning framework for few-shot function
   prediction with uncertainty quantification.
2. **Flow Matching** provides a simple, stable training objective (MSE!) for
   learning expressive generative models.

By using flow matching as the predictive mechanism, FlowNP overcomes the
Gaussian output limitation of classical NPs, enabling it to represent
**arbitrary conditional distributions** over function values.

### Architecture Summary

| Component | Implementation | Purpose |
|-----------|---------------|---------|
| Fourier encoding | `fourier_encode` — sinusoidal with geometric frequencies | Lift low-dim coordinates to high-dim features |
| Time conditioning | Append scalar $t$ to spatial coords before encoding | Enable the model to distinguish noise levels |
| Token embedding | 4-layer ReLU MLP | Mix Fourier features and project to $d_\text{model}$ |
| Transformer encoder | 6 layers, 4 heads, bidirectional | Learn context-target interactions |
| Velocity head | 2-layer MLP | Predict the velocity $v_\theta$ from encoded representations |

### Training

- **Loss**: Conditional flow matching MSE — regress velocity against $y_1 - y_0$
- **Optimiser**: Adam with cosine annealing LR schedule
- **Data**: Random context/target splits with variable-length padding and masking

### Inference

- **Method**: Stochastic Euler integration from Gaussian noise to predictions
- **Coefficients**: $\alpha(t) = 1 + t(1-t)$, $\sigma(t) = 0.2\sqrt{t(1-t)}$
- **Output**: Mean and standard deviation from $S$ independent sample trajectories

### Running experiments

```bash
cd flownp_spl
python train.py --config configs/weather.yaml --device cuda:0   # PeakWeather
python train.py --config configs/gdp.yaml     --device cuda:1   # GDP
```

All hyperparameters live in the YAML configs — no code changes needed.